# Facet, notebook 2: RoBERTa-base, ONNX int8, ready for the browser



In [2]:
!pip install -q -U transformers datasets accelerate sentencepiece seaborn "optimum[onnxruntime]" onnx onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 106.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 77.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 12.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages tha

In [3]:
import os, json, numpy as np, pandas as pd, torch
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient

login(UserSecretsClient().get_secret("HF_TOKEN"))      
api = HfApi()

LABELS = ["negative", "neutral", "positive"]
l2i = {l: i for i, l in enumerate(LABELS)}

def ece(conf, correct, bins=10):
    edges = np.linspace(0, 1, bins + 1); total = 0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return total

REPO = "sy12ssss/absa-roberta-base"
MODEL = "roberta-base"
REPORT_TO = "none"
os.makedirs("results", exist_ok=True)

## 1. Data (same splits as notebook 1)

In [4]:
from datasets import load_dataset

frames = []
for repo, dom in [("tomaarsen/setfit-absa-semeval-restaurants", "restaurant"),
                  ("tomaarsen/setfit-absa-semeval-laptops", "laptop")]:
    ds = load_dataset(repo)
    for split in ds.keys():
        df = ds[split].to_pandas()
        lab = ds[split].features.get("label")
        if hasattr(lab, "names"):
            df["label"] = df["label"].map(dict(enumerate(lab.names)))
        df["domain"] = dom
        frames.append(df)
    print(dom, {k: len(v) for k, v in ds.items()})

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/147k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/46.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3693 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1134 [00:00<?, ? examples/s]

...

data/train-00000-of-00001.parquet:   0%|          | 0.00/117k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/30.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2358 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/654 [00:00<?, ? examples/s]

laptop {'train': 2358, 'test': 654}


In [5]:
from sklearn.model_selection import GroupShuffleSplit

full_df = pd.concat(frames, ignore_index=True)
full_df = full_df.rename(columns={c: "aspect" for c in ("span", "term", "aspect_term") if c in full_df.columns})
full_df["label"] = full_df["label"].astype(str).str.strip().str.lower()
full_df = full_df[full_df["label"].isin(l2i)].copy()         
full_df["label"] = full_df["label"].map(l2i)
full_df = (full_df[["text", "aspect", "label", "domain"]]
           .drop_duplicates(subset=["text", "aspect"]).reset_index(drop=True))
print("Total (aspect, sentence) pairs:", len(full_df))
assert len(full_df) == 5822, "Unexpected pair count: check the data cell output"

def group_split(df, test_size, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    a, b = next(gss.split(df, groups=df["text"]))          
    return df.iloc[a].reset_index(drop=True), df.iloc[b].reset_index(drop=True)

train_df, tmp_df = group_split(full_df, 0.2)
val_df, test_df = group_split(tmp_df, 0.5)
print("train/val/test:", len(train_df), len(val_df), len(test_df))    

Total (aspect, sentence) pairs: 5822
train/val/test: 4652 582 588


## 2. Tokenize and fine-tune

In [6]:
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, set_seed)

tok = AutoTokenizer.from_pretrained(MODEL)

def to_ds(df):
    d = df[["text", "aspect", "label"]].reset_index(drop=True)
    return Dataset.from_pandas(d).map(
        lambda b: tok(b["aspect"], b["text"], truncation=True, max_length=128), batched=True)

train_ds, val_ds, test_ds = map(to_ds, (train_df, val_df, test_df))

set_seed(42)
os.environ["MLFLOW_EXPERIMENT_NAME"] = "absa-sentiment"
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3, id2label=dict(enumerate(LABELS)), label2id=l2i).float()   
print("param dtype:", next(model.parameters()).dtype)

def compute_metrics(p):
    pr = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, pr), "f1_macro": f1_score(p.label_ids, pr, average="macro")}

steps_per_epoch = -(-len(train_ds) // 16)
args = TrainingArguments(
    output_dir="out", learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=4, warmup_steps=int(0.1 * steps_per_epoch * 4), weight_decay=0.01,
    fp16=True, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1_macro",
    save_total_limit=1, report_to=REPORT_TO, seed=42)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


param dtype: torch.float32


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.588383,0.754296,0.655381
2,0.638400,0.501270,0.817869,0.745635
3,0.638400,0.501484,0.833333,0.758624
4,0.271400,0.610149,0.847079,0.791076


TrainOutput(global_step=1164, training_loss=0.41507504649997984, metrics={'train_runtime': 110.7769, 'train_samples_per_second': 167.977, 'train_steps_per_second': 10.508, 'total_flos': 526235057750304.0, 'train_loss': 0.41507504649997984, 'epoch': 4.0})

## 3. Evaluation and calibration (PyTorch fp32)

In [7]:
from sklearn.metrics import classification_report
from scipy.special import softmax, log_softmax
from scipy.optimize import minimize_scalar

out = trainer.predict(test_ds); yt, Lt = out.label_ids, out.predictions
report = classification_report(yt, Lt.argmax(-1), target_names=LABELS, digits=4)
print("test size:", len(yt)); print(report)
open("results/classification_report_roberta.txt", "w").write(report)

val_out = trainer.predict(val_ds); Lv, yv = val_out.predictions, val_out.label_ids
nll = lambda T: -log_softmax(Lv / T, axis=-1)[np.arange(len(yv)), yv].mean()
T = float(minimize_scalar(nll, bounds=(0.5, 5.0), method="bounded").x)
p0, p1 = softmax(Lt, -1), softmax(Lt / T, -1)
ok = (p0.argmax(-1) == yt).astype(float)
print("T =", round(T, 3), "| ECE before/after:", round(ece(p0.max(-1), ok), 4), round(ece(p1.max(-1), ok), 4))

test size: 588
              precision    recall  f1-score   support

    negative     0.7907    0.7640    0.7771       178
     neutral     0.5619    0.5221    0.5413       113
    positive     0.8810    0.9226    0.9013       297

    accuracy                         0.7976       588
   macro avg     0.7445    0.7362    0.7399       588
weighted avg     0.7924    0.7976    0.7945       588



T = 1.979 | ECE before/after: 0.1538 0.0664


## 4. Push the PyTorch model

In [8]:
OUT = "rob_model"
trainer.model.save_pretrained(OUT); tok.save_pretrained(OUT)
cp = f"rob_model/tokenizer_config.json"; c = json.load(open(cp))
if isinstance(c.get("extra_special_tokens"), list):          
    c["extra_special_tokens"] = {}; json.dump(c, open(cp, "w"), indent=2)
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=REPO)

AutoTokenizer.from_pretrained(REPO, force_download=True)
print("pushed and reloaded OK:", "https://huggingface.co/" + REPO)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

pushed and reloaded OK: https://huggingface.co/sy12ssss/absa-roberta-base


## 5. Export to ONNX and quantize to int8

In [9]:
!optimum-cli export onnx --model rob_model --task text-classification onnx_rob_fp32

from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

ORTQuantizer.from_pretrained("onnx_rob_fp32").quantize(
    save_dir="onnx_rob_int8",
    quantization_config=AutoQuantizationConfig.avx2(is_static=False, per_channel=False))

import onnxruntime as ort
for p in ["onnx_rob_fp32/model.onnx", "onnx_rob_int8/model_quantized.onnx"]:
    print(p, round(os.path.getsize(p) / 1e6), "MB")
print("inputs:", [i.name for i in ort.InferenceSession("onnx_rob_int8/model_quantized.onnx").get_inputs()])

Multiple distributions found for package optimum. Picked distribution: optimum-onnx
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


Multiple distributions found for package optimum. Picked distribution: optimum-onnx
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


onnx_rob_fp32/model.onnx 499 MB
onnx_rob_int8/model_quantized.onnx 125 MB
inputs: ['input_ids', 'attention_mask']


## 6. Verify that int8 did not lose accuracy

Do not push a quantized model without this check: the DeBERTa int8 model looked fine but was near chance.

In [10]:
from sklearn.metrics import accuracy_score, f1_score

def run(path, df, bs=32):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    names = {i.name for i in sess.get_inputs()}
    out = []
    for i in range(0, len(df), bs):
        b = df.iloc[i:i + bs]
        e = tok(list(b.aspect), list(b.text), return_tensors="np", padding=True,
                truncation="only_second", max_length=128)
        out.append(sess.run(None, {k: v for k, v in e.items() if k in names})[0])
    return np.concatenate(out)

res = {}
for label, f in [("fp32", "onnx_rob_fp32/model.onnx"), ("int8", "onnx_rob_int8/model_quantized.onnx")]:
    pr = run(f, test_df).argmax(-1)
    res[label] = (accuracy_score(yt, pr), f1_score(yt, pr, average="macro"))
    print(f"{label}: {round(os.path.getsize(f) / 1e6)} MB | acc {res[label][0]:.4f} | macro-F1 {res[label][1]:.4f}")
assert res["int8"][0] >= res["fp32"][0] - 0.02, "int8 lost more than 2 points: do NOT push it"

fp32: 499 MB | acc 0.7976 | macro-F1 0.7399
int8: 125 MB | acc 0.8044 | macro-F1 0.7504


## 7. Recalibrate the int8 model and push the ONNX files

In [11]:
Lv8 = run("onnx_rob_int8/model_quantized.onnx", val_df)
Lt8 = run("onnx_rob_int8/model_quantized.onnx", test_df)
yv = val_df["label"].values

nll8 = lambda T: -log_softmax(Lv8 / T, axis=-1)[np.arange(len(yv)), yv].mean()
T8 = float(minimize_scalar(nll8, bounds=(0.5, 5.0), method="bounded").x)
q0, q1 = softmax(Lt8, -1), softmax(Lt8 / T8, -1)
ok8 = (q0.argmax(-1) == yt).astype(float)
print("T (int8) =", round(T8, 3), "| ECE before/after:", round(ece(q0.max(-1), ok8), 4), round(ece(q1.max(-1), ok8), 4))
assert T8 < 4.9, "temperature at the search bound: something is wrong, do not push"

json.dump({"temperature": T8}, open("results/calibration_onnx.json", "w"))
api.upload_file(path_or_fileobj="onnx_rob_int8/model_quantized.onnx", path_in_repo="onnx/model_quantized.onnx", repo_id=REPO)
api.upload_file(path_or_fileobj="results/calibration_onnx.json", path_in_repo="calibration_onnx.json", repo_id=REPO)
api.upload_file(path_or_fileobj="results/classification_report_roberta.txt", path_in_repo="classification_report.txt", repo_id=REPO)
print(api.list_repo_files(REPO))

T (int8) = 1.913 | ECE before/after: 0.1377 0.0647


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


['.gitattributes', 'calibration.json', 'calibration_onnx.json', 'classification_report.txt', 'config.json', 'merges.txt', 'model.safetensors', 'onnx/model_quantized.onnx', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']
